# PoliMillionaire - BM25 (SimpleWiki + KELM) con Reranking BERT (No LLM)

Questa pipeline usa BM25 come recuperatore iperveloce. I 20 migliori risultati fusi da SimpleWiki e KELM vengono passati ad un **Cross-Encoder BERT** che per ogni opzione legge e giudica la correttezza del RAG semantico, assegnando uno score da 0 a 10 senza generare testo alcuno.

In [1]:
from pathlib import Path
import os
import sys
import numpy as np

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "api_client" / "NLP_assignment_api_client").exists() and (path / "project" / "src").exists():
            return path
    raise FileNotFoundError("Could not find project root")

PROJECT_ROOT = find_project_root()
CLIENT_PARENT = PROJECT_ROOT / "api_client" / "NLP_assignment_api_client"
SRC_DIR = PROJECT_ROOT / "project" / "src"

for path in [str(CLIENT_PARENT), str(SRC_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\Tommaso\Code\NLP_polimi_26


In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
# Bypass per shm.dll mancante o corrotta nel path conda
try:
    import ctypes
    ctypes.cdll.LoadLibrary(
        os.path.join(os.path.dirname(torch.__file__), 'lib', 'shm.dll')
    )
except Exception:
    pass

from getpass import getpass
from millionaire_client import MillionaireClient
from retrieval_quiz_runner import load_retrieval_index, retrieve, compact_snippet, RetrievalDecision, run_all_competitions, summarize, write_logs
from sentence_transformers import CrossEncoder

API_URL = os.getenv("POLIMILLIONAIRE_API_URL", "http://131.175.15.22:51111/")
USERNAME = os.getenv("POLIMILLIONAIRE_USERNAME") or input("Username: ").strip()
PASSWORD = os.getenv("POLIMILLIONAIRE_PASSWORD") or getpass("Password: ")

client = MillionaireClient(API_URL, timeout=30)
user = client.login(USERNAME, PASSWORD)
print(f"Logged in as {user.username} ({user.role})")

OSError: [WinError 127] Impossibile trovare la procedura specificata. Error loading "c:\Users\Tommaso\conda-envs\polimillionaire\Lib\site-packages\torch\lib\shm.dll" or one of its dependencies.

In [ ]:
WIKI_INDEX_PATH = PROJECT_ROOT / "data" / "indexes" / "simplewiki_160w_bm25.joblib"
KELM_INDEX_PATH = PROJECT_ROOT / "data" / "indexes" / "kelm_500k_bm25.joblib"

wiki_index = load_retrieval_index(WIKI_INDEX_PATH)
kelm_index = load_retrieval_index(KELM_INDEX_PATH)
indexes = [wiki_index, kelm_index]

In [ ]:
# Carichiamo il modello Cross-Encoder leggerissimo (occupa pochissima VRAM/RAM)
bert_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)

def choose_con_bert(question, index, top_k_bm25=10):
    """Una funzione custom da passare al runner per usare BERT al posto dell'euretica base."""
    
    # 1) Fase Retriever: Recuperiamo i top documenti da BM25 (RRF fusi)
    global_query = f"{question.text}"
    docs_candidati = retrieve(index, global_query, top_k=top_k_bm25)
    
    # Costruiamo il paragrafo unico di contesto
    context_text = " ".join([doc.get('title', '') + ": " + doc.get('text', '') for _, doc in docs_candidati])
    
    option_scores = []
    for opt in question.options:
        # 2) Fase Reranker: Creiamo la coppia (Contesto, Domanda + Opzione)
        ipotesi = f"{question.text} La risposta corretta è {opt.text}."
        
        # Applichiamo BERT alla coppia senza generare testo
        bert_score = float(bert_model.predict([context_text, ipotesi])[0])
        
        option_scores.append({
            "option_id": opt.id,
            "option_text": str(opt.text),
            "score": bert_score,
            "evidence_score": bert_score, # Teniamo in log il valore di BERT
        })

    # Ordiniamo in base al giudizio di BERT
    option_scores.sort(key=lambda row: row["score"], reverse=True)
    best = option_scores[0]
    second_score = option_scores[1]["score"] if len(option_scores) > 1 else 0.0
    confidence = best["score"] - second_score

    return RetrievalDecision(
        option_id=int(best["option_id"]),
        option_text=str(best["option_text"]),
        confidence=float(confidence),
        option_scores=option_scores,
        evidence=[{"score": sc, **compact_snippet(dc)} for sc, dc in docs_candidati]
    )

In [ ]:
ATTEMPTS_PER_COMPETITION = 5
STRATEGY_NAME = "bm25_mixed_plus_bert_reranker"

rows = run_all_competitions(
    client=client,
    index=indexes,
    strategy_name=STRATEGY_NAME,
    attempts_per_competition=ATTEMPTS_PER_COMPETITION,
    choose_fn=choose_con_bert, # <--- Magia: iniettiamo la nostra funzione BERT custom senza toccare lo script master!
)

output_path = PROJECT_ROOT / "logs" / "simplewiki_kelm_bert_no_llm_all_competitions.csv"
write_logs(rows, output_path)
print(f"Wrote {len(rows)} rows to {output_path}")

In [ ]:
summarize(rows)